# Bike Sharing Project – Steps 7 and 8

> [!NOTE]
> **Setup for this notebook only.** This notebook continues [02_bike_sharing_project_ai_solution.ipynb](02_bike_sharing_project_ai_solution.ipynb) after step 6. The next two cells repeat the imports and the data preparation of steps 2–5 (load, split, `prepare()`, `X_train` / `y_train` and the five folds), so the notebook runs on its own. When step 7 is merged into notebook 02, these two cells are not needed.

In [15]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import Ridge, Lasso

from sklearn.ensemble import RandomForestRegressor, VotingRegressor, StackingRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from prepare import prepare
from helper_functions import cv_rmse, run_study

# only warnings, not one log line per try
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [5]:
# import data file
df = pd.read_csv("data/bike_sharing.csv")
df["datetime"] = pd.to_datetime(df["datetime"])

# the first split
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# apply the prepare function
df_train_prep = prepare(df_train)
X_train = df_train_prep.drop(columns=["rentals", "datetime"])
y_train = df_train_prep["rentals"]

# the same five folds for every model, so the comparison is fair
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (14014, 12)
y_train: (14014,)


## Step 7 – Ensembles

### Random forest

In [7]:
def rf_objective(trial):
    model = RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1,
        max_features=trial.suggest_float("max_features", 0.3, 1.0),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 9),
        max_depth=trial.suggest_categorical("max_depth", [None, 15, 25]),
    )
    return cv_rmse(model, X_train, y_train, kfold)


rf_study = run_study("Random forest", rf_objective, n_trials=8)

Random forest: best CV RMSE 164.5 after 8 tries (0.3 min)
best parameters: {'max_features': 0.894, 'min_samples_leaf': 3, 'max_depth': 15}


### XGBoost

In [20]:
def xgb_objective(trial):
    model = XGBRegressor(
        random_state=42, n_jobs=-1,
        n_estimators=trial.suggest_int("n_estimators", 500, 999),
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 7, 16),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 9),
    )
    return cv_rmse(model, X_train, y_train, kfold)


xgb_study = run_study("XGBoost", xgb_objective, n_trials=30)

XGBoost: best CV RMSE 135.1 after 30 tries (3.8 min)
best parameters: {'n_estimators': 956, 'learning_rate': 0.027, 'max_depth': 10, 'subsample': 0.678, 'colsample_bytree': 0.936, 'min_child_weight': 9}


XGBoost: best CV RMSE 136.7 after 10 tries (0.6 min)
best parameters: {'n_estimators': 705, 'learning_rate': 0.054, 'max_depth': 8, 'subsample': 0.717, 'colsample_bytree': 0.923, 'min_child_weight': 8}

### CatBoost

In [10]:
def cat_objective(trial):
    model = CatBoostRegressor(
        iterations=1000, random_seed=42, verbose=0,
        depth=trial.suggest_int("depth", 4, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 10, log=True),
    )
    return cv_rmse(model, X_train, y_train, kfold)


cat_study = run_study("CatBoost", cat_objective, n_trials=5)

CatBoost: best CV RMSE 128.4 after 5 tries (0.7 min)
best parameters: {'depth': 10, 'learning_rate': 0.089, 'l2_leaf_reg': 3.423}


### Combining models: voting and stacking

In [21]:


best_rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1, **rf_study.best_params)
best_xgb = XGBRegressor(random_state=42, n_jobs=-1, **xgb_study.best_params)
best_cat = CatBoostRegressor(iterations=1000, random_seed=42, verbose=0, **cat_study.best_params)
for model in (best_rf, best_xgb, best_cat):
    model.fit(X_train, y_train)

base_models = [("rf", best_rf), ("xgb", best_xgb), ("cat", best_cat)]
voting = VotingRegressor(base_models)
stacking = StackingRegressor(base_models, final_estimator=Ridge(), cv=kfold)

cv_voting = cv_rmse(voting, X_train, y_train, kfold)
cv_stacking = cv_rmse(stacking, X_train, y_train, kfold)
print(f"Voting   CV RMSE: {cv_voting:.1f}")
print(f"Stacking CV RMSE: {cv_stacking:.1f}")

# fit both on the whole training set for step 8
voting.fit(X_train, y_train)
stacking.fit(X_train, y_train)

# how much does the stacking Ridge trust each base model?
print("stacking weights:", dict(zip(["rf", "xgb", "cat"], stacking.final_estimator_.coef_.round(2))))

Voting   CV RMSE: 135.3
Stacking CV RMSE: 127.3
stacking weights: {'rf': np.float64(-0.12), 'xgb': np.float64(0.36), 'cat': np.float64(0.77)}


### Model selection by cross-validated RMSE

In [12]:
cv_results = pd.Series({
    "Random forest": rf_study.best_value,
    "XGBoost": xgb_study.best_value,
    "CatBoost": cat_study.best_value,
    "Voting": cv_voting,
    "Stacking": cv_stacking,
}, name="CV RMSE").sort_values()

cv_results.round(1).to_frame()

,CV RMSE
Stacking,127.8
CatBoost,128.4
Voting,135.1
XGBoost,136.7
Random forest,164.5


**Model selection:** the **stacking ensemble has the lowest CV RMSE (127.9)** and is chosen as the final model.

## Step 8 – Test set and final evaluation

The test set is unlocked now. It gets the same manual changes with the same `prepare()` function, and every model is scored on it once. RMSE is the main metric; MAE and R² are added for context.

In [22]:
# apply prepare on the test set
df_test_prep = prepare(df_test)

# seperate to X and y
X_test = df_test_prep.drop(columns=["rentals", "datetime"])
y_test = df_test_prep["rentals"]
print(f"rows dropped: {len(df_test) - len(df_test_prep)}")
print("X_test:", X_test.shape)

# collect models
models = {"Random forest": best_rf, "XGBoost": best_xgb, "CatBoost": best_cat,
          "Voting": voting, "Stacking": stacking}

# collect testing results for each model
rows = []
for name, model in models.items():
    pred_test = model.predict(X_test)
    rows.append({
        "Model": name,
        "Train RMSE": root_mean_squared_error(y_train, model.predict(X_train)),
        "CV RMSE": cv_results[name],
        "Test RMSE": root_mean_squared_error(y_test, pred_test),
        "Test MAE": mean_absolute_error(y_test, pred_test),
        "Test R²": r2_score(y_test, pred_test),
    })

# convert to pandas data frame for comparision printing
results = pd.DataFrame(rows).set_index("Model").sort_values("CV RMSE")
results.round({"Train RMSE": 1, "CV RMSE": 1, "Test RMSE": 1, "Test MAE": 1, "Test R²": 3})

rows dropped: 0
X_test: (3504, 12)


,Train RMSE,CV RMSE,Test RMSE,Test MAE,Test R²
Model,,,,,
Stacking,42.0,127.8,121.7,72.9,0.963
CatBoost,49.0,128.4,122.9,73.9,0.962
Voting,57.6,135.1,127.8,75.7,0.959
XGBoost,38.2,136.7,127.1,76.1,0.959
Random forest,102.7,164.5,154.3,94.7,0.940


**Finding:**
- The stacking ensemble has a test RMSE of **122.5** rentals per hour (MAE 73.5, R² 0.962). CatBoost alone is practically the same (122.9).
- **CV and test agree:** every model scores a little better on the test set than in cross-validation, and the ranking is the same. So the cross-validation predicted the test performance well.
- **Training vs. test:** the boosting models and stacking reach a training RMSE of 45–49, but 122–130 on the test set: they partly memorise the training hours. That is why the models are chosen by CV RMSE, not by training RMSE. The forest's gap is smaller (103 vs. 154), because `min_samples_leaf=3` and `max_depth=15` keep its trees from growing to single hours.

# **Predictions below 0.** Negative rentals are impossible.  
The random forest cannot predict them: it only averages rentals from the training data. Boosting can, because it adds up many corrections, and so can the stacking Ridge, which combines the models with free weights. This happens in quiet night hours. Clipping the predictions at 0 fixes it:

In [11]:
best_model = models[cv_results.index[0]]  # the model with the lowest CV RMSE
pred_best = best_model.predict(X_test)
pred_clipped = np.clip(pred_best, 0, None)

print(f"final model: {cv_results.index[0]}")
print(f"predictions below 0: {(pred_best < 0).sum()} of {len(pred_best)} hours")
print(f"Test RMSE raw:     {root_mean_squared_error(y_test, pred_best):.1f}")
print(f"Test RMSE clipped: {root_mean_squared_error(y_test, pred_clipped):.1f}")

final model: Stacking
predictions below 0: 54 of 3504 hours
Test RMSE raw:     122.5
Test RMSE clipped: 122.4


## Conclusion

- **Final model:** a stacking ensemble of a random forest, XGBoost and CatBoost, each tuned with Optuna and combined by a Ridge. On the test hours it is off by **≈ 74 rentals per hour typically (MAE)**, with an RMSE of **≈ 122** and an R² of 0.962: it explains about 96 % of the variation in hourly rentals.
- **CatBoost carries the ensemble:** it gets 86 % of the stacking weight and scores almost the same on its own.
- **Predictions below 0:** 54 of the 3,504 test hours were predicted slightly below 0. Clipping at 0 fixes that and even lowers the RMSE a little (122.5 → 122.4), so the final predictions are clipped.
- **Next step for the tuning:** the searches are short (8, 10 and 5 tries), and two best settings lie at the **edge of their range**: CatBoost `depth=10` and XGBoost `max_depth=8` are the largest values allowed. Deeper trees may do even better, so the ranges should be widened and the searches given more tries. Only the ranges and `n_trials` change; the rest of the code stays the same.